# Predictive Circuit Coding Training

This notebook is the Colab-first training surface. It installs the repo first, then uses package helpers for stage banners, preflight checks, and checkpoint messaging.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
repo_root = Path('/content/predictive-circuit-coding')
drive_root = Path('/content/drive/MyDrive/pcc_runs')
drive_root.mkdir(parents=True, exist_ok=True)
print('Drive mounted. Repo root:', repo_root)

In [ ]:
if not repo_root.exists():
    !git clone https://github.com/jacobposchl/predictive-circuit-coding.git {repo_root}
%cd {repo_root}
!pip install -e .[notebook]

In [ ]:
from predictive_circuit_coding.utils import NotebookStageReporter, verify_paths_exist

reporter = NotebookStageReporter(name='colab-train', expected_duration='2-10 minutes for debug runs, longer for full A100 runs')
reporter.banner('Predictive Circuit Coding Training', subtitle='Setup, preflight, training, and evaluation')
experiment_config = repo_root / 'configs' / 'pcc' / 'predictive_circuit_coding_base.yaml'
data_config = repo_root / 'configs' / 'pcc' / 'allen_visual_behavior_neuropixels_local.yaml'
checkpoint_dir = repo_root / 'artifacts' / 'checkpoints'
resume_checkpoint = checkpoint_dir / 'pcc_best.pt'

In [ ]:
reporter.begin('preflight', next_artifact='best checkpoint + evaluation summary')
paths_ok = verify_paths_exist({
    'experiment_config': experiment_config,
    'data_config': data_config,
})
print(paths_ok)
assert all(paths_ok.values()), 'Missing config files for training notebook.'
print('Checkpoint resume path:', resume_checkpoint)
print('If you want to resume, point training.resume_checkpoint in the experiment config at an existing checkpoint.')
reporter.finish('preflight')

In [ ]:
reporter.begin('training', next_artifact='artifacts/checkpoints/pcc_best.pt')
%cd {repo_root}
!pcc-train --config {experiment_config} --data-config {data_config}
reporter.note_checkpoint(resume_checkpoint)
reporter.finish('training')

In [ ]:
reporter.begin('evaluation', next_artifact='validation-ready evaluation summary json')
%cd {repo_root}
!pcc-evaluate --config {experiment_config} --data-config {data_config} --checkpoint {resume_checkpoint} --split valid
reporter.finish('evaluation')